# Evaluation — Pharasiot 

Functions are in `evaluation_functions.py`. This just calls them.

In [ ]:
import os
import pandas as pd

from evaluation_functions import *

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.precision", 3)

RESULTS_FOLDER = "evaluations"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

print(len(TASKS), "tasks,", len(MODELS), "models,", len(SEEDS), "orderings")

## Check the files first

Every file should have the right number of items and a letter for each. The
correct letter of an item should be the same in Pharasiot and in Greek, because
the same permutation is applied to both. Expected: 78 files checked, 0 problems.

In [ ]:
# check the files first: right number of items, a letter for each,
# and the same answer key in Pharasiot and Greek
problems = 0
n_files = 0

for task in TASKS:
    for model in MODELS:
        for seed in SEEDS:
            keys = {}
            for condition in conditions(task):
                answers = load_results(task, condition, seed, model)
                n_files += 1

                no_letter = [item for item, (chosen, key) in answers.items()
                             if chosen not in LETTERS]
                if len(answers) != n_items(task) or no_letter:
                    problems += 1
                    print("CHECK", task, condition, model, seed, "-", len(answers),
                          "items; no letter on:", no_letter)

                keys[condition] = {item: key for item, (chosen, key) in answers.items()}

            if has_greek(task) and keys["pharasiot"] != keys["greek"]:
                problems += 1
                print("CHECK", task, model, seed,
                      "- the correct letters differ between the conditions")

print(n_files, "files checked,", problems, "problems")

# Step 1 — descriptive

Counting, no statistics yet.

## Accuracy

`chance` = guessing. `majority` = always answering the most common label,
which only exists for sentiment and topic.

In [ ]:
accuracy = pd.DataFrame(table_accuracy())
accuracy

In [ ]:
accuracy.pivot_table(index=["task", "condition"],
                     columns="model",
                     values="accuracy")

In [ ]:
# what didn't beat guessing
accuracy[~accuracy.beats_chance]

## Macro-F1

Sentiment and topic only. Scores each label separately, averages them equally,
so a small failed label still shows.

Close to accuracy = even performance. A gap = one label is failing.

In [ ]:
macro = pd.DataFrame(table_macro_f1())

# side by side, to see the gap
macro_vs_accuracy = macro.merge(
    accuracy[["task", "condition", "seed", "model", "accuracy"]],
    on=["task", "condition", "seed", "model"])
macro_vs_accuracy["gap"] = (macro_vs_accuracy.accuracy
                            - macro_vs_accuracy.macro_f1)
macro_vs_accuracy

In [ ]:
per_class = pd.DataFrame(table_per_class())

# check n_gold against n_predicted
per_class[per_class.task == "sentiment"]

In [ ]:
per_class[per_class.task == "topic"]

## Position bias

Drops the items where the model's favourite letter was correct, then rescores.
Below `chance` means the score came from the habit.

In [ ]:
bias = pd.DataFrame(table_position_bias())

bias["survives"] = bias.filtered_accuracy > bias.chance
bias

In [ ]:
# score came from the letter, not from reading
bias[~bias.survives]

# Step 2 — significance tests

## Binomial — better than guessing?

Orderings pooled, so the same items appear twice and the p-values are a little
optimistic. Worth mentioning in the write-up.

In [ ]:
binomial = pd.DataFrame(table_binomial())
binomial

In [ ]:
binomial[~binomial.beats_chance]

## McNemar — is the gap real?

Only counts items where the conditions disagreed. Even split = no effect.

**underpowered** = fewer than six disagreements, so nothing could have reached
significance. Not the same as no effect.

`krikri+meltemi` leaves Llama out.

In [ ]:
mcnemar = pd.DataFrame(table_mcnemar())
mcnemar

In [ ]:
mcnemar[mcnemar.model == "krikri+meltemi"]

# Step 3 — dialect cost

## Per task

Gap in items, in points, and as retention.

`neither` = failed in Greek too, so not the dialect's fault.

In [ ]:
cost = pd.DataFrame(table_dialect_cost())
cost

In [ ]:
cost.pivot_table(index="task", columns="model", values="gap_points")

## By level

word = lexical · sentence = sentiment, topic, NLI · paragraph = factual,
context tracking, discourse

In [ ]:
by_level = pd.DataFrame(table_by_level())
by_level

## Overall

Retention uses only the six tasks with a Greek control.

In [ ]:
overall = pd.DataFrame(table_overall())
overall

# Save

In [ ]:
tables = {
    "accuracy": accuracy,
    "macro_f1": macro_vs_accuracy,
    "per_class": per_class,
    "position_bias": bias,
    "binomial": binomial,
    "mcnemar": mcnemar,
    "dialect_cost": cost,
    "by_level": by_level,
    "overall": overall,
}

for name, table in tables.items():
    path = os.path.join(RESULTS_FOLDER, name + ".tsv")
    table.to_csv(path, sep="\t", index=False)
    print("wrote", path)

# everything in one readable file
report = format_report(tables)
path = os.path.join(RESULTS_FOLDER, "report.txt")
with open(path, "w", encoding="utf-8") as f:
    f.write(report)
print("wrote", path)